# NNUE-PyTorch Training in Google Colab
### Fork: `https://github.com/wheres-perry/nnue-pytorch`

This notebook provides a complete environment to train Stockfish NNUE networks with the new **`LatentThreats`** feature set (`SFNNv16_Latent`) on a Google Colab GPU instance.

**Pipeline Overview:**
1. GPU & CUDA verification
2. Clone the repository and install requirements
3. Build the native C++ data loader (`libtraining_data_loader.so`) with PGO
4. Verify the `LatentThreats` composed architecture (99,184 real inputs, 296 max active)
5. Launch training with live TensorBoard
6. Serialize the trained model to `.nnue` for Stockfish evaluation
7. (Optional) Generate temporary depth-15 Stockfish training data and train

## 1. GPU & CUDA Verification
Ensure you have a GPU allocated in Colab (Go to `Runtime` -> `Change runtime type` -> select `T4 GPU`, `V100`, or `A100`).

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")

## 2. Clone Fork & Install Dependencies

In [ ]:
# Clone your fork of nnue-pytorch
!git clone https://github.com/wheres-perry/nnue-pytorch.git
%cd nnue-pytorch

# Install project dependencies
!pip install -q -r requirements.txt
!pip install -q tyro schedulefree torchmetrics asciimatics

## 3. Compile Native C++ Data Loader (PGO)
Compiles `training_data_loader` with Profile-Guided Optimization using the included small binpack dataset.

In [ ]:
# Build the shared library with PGO
!bash compile_data_loader.sh

# Verify that libtraining_data_loader.so was created
!ls -lh libtraining_data_loader.so

## 4. Verify Feature Set & Architecture Dimensions
Runs a quick sanity check verifying that `LatentThreats` is recognized and the composed model has the exact expected dimensions (`99,184` real inputs, `296` max active features).

In [ ]:
import torch
from model.modules.features import LatentThreats, get_feature_cls, get_available_features
from model.modules import ComposedFeatureTransformer
from model.quantize import QuantizationConfig, QuantizationManager

print("Available features in factory:", get_available_features())
assert "LatentThreats" in get_available_features(), "LatentThreats not found!"

feature_str = "Full_Threats+PP_3Wide+HalfKAv2_hm^+LatentThreats"
classes = get_feature_cls(feature_str)

q_mgr = QuantizationManager(QuantizationConfig())
cft = ComposedFeatureTransformer(classes, 1024, 8, q_mgr)

print("\n--- Feature Transformer Statistics ---")
print(f"Feature Composition: {cft.FEATURE_NAME}")
print(f"Total Inputs (with virtual): {cft.NUM_INPUTS}")
print(f"Real Weights (for export):   {cft.NUM_REAL_FEATURES}  (Expected: 99,184)")
print(f"Max Active Features:         {cft.MAX_ACTIVE_FEATURES}  (Expected: 296)")
print(f"Feature Hash:                {hex(cft.HASH)}")

assert cft.NUM_REAL_FEATURES == 99184
assert cft.MAX_ACTIVE_FEATURES == 296
print("\nVerification PASSED!")

## 5. (Optional) Mount Google Drive
Mount Google Drive to persist training checkpoints or to read training datasets (`.binpack`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 6. Live TensorBoard Monitoring

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./logs

## 7. Run Training
Specify your dataset path (`.binpack`). You can train on your own data or use the sample binpack for testing.

Adjust hyperparameters:
- `--features "Full_Threats+PP_3Wide+HalfKAv2_hm^+LatentThreats"`: Includes the new 1-blocker alignment features.
- `--batch-size 16384`: Typical batch size for 16GB+ VRAM GPU. Adjust to `8192` if running on standard T4.
- `--max-epochs 800`: Total training epochs.
- `--default-root-dir`: Directory for checkpoints and TensorBoard logs.

In [ ]:
# Set your training dataset path:
# e.g. DATASET_PATH = "/content/drive/MyDrive/chess/training_data.binpack"
DATASET_PATH = ".pgo/small.binpack"  # Using included sample binpack for smoke testing

!python train.py {DATASET_PATH} \
    --features "Full_Threats+PP_3Wide+HalfKAv2_hm^+LatentThreats" \
    --batch-size 16384 \
    --max-epochs 50 \
    --threads 4 \
    --default-root-dir ./logs/latent_threats_run

## 8. Export / Serialize to Stockfish `.nnue`
Converts the trained PyTorch checkpoint into a serialized `.nnue` file that can be loaded directly into Stockfish (`eval.nnue`).

In [ ]:
import glob

checkpoints = sorted(glob.glob("./logs/latent_threats_run/**/checkpoints/*.ckpt", recursive=True))
if checkpoints:
    latest_ckpt = checkpoints[-1]
    print(f"Latest checkpoint: {latest_ckpt}")
    !python serialize.py "{latest_ckpt}" "./nn-latent.nnue"
    print("\nSerialized network successfully saved to ./nn-latent.nnue!")
else:
    print("No checkpoints found. Ensure training has completed at least one checkpoint epoch.")

## 9. (Optional) Generate Depth-15 Stockfish Training Data & Train
If you do not have a pre-existing `.binpack` dataset, this section installs Stockfish, plays sample games, evaluates positions at depth 15, packs them into `custom_d15.binpack`, and trains on them.

In [ ]:
# 1. Install Stockfish and python-chess
!apt-get update -qq && apt-get install -y -qq stockfish
!pip install -q python-chess

import os
import random
import chess
import chess.engine

# 2. Generate games & evaluate positions at depth 15
NUM_GAMES = 20
POSITIONS_FILE = "generated_positions.txt"
STOCKFISH_PATH = "/usr/games/stockfish"

print(f"Starting Stockfish from {STOCKFISH_PATH} at depth 15...")
engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)

total_positions = 0
with open(POSITIONS_FILE, "w") as f:
    for game_idx in range(NUM_GAMES):
        board = chess.Board()
        game_fens = []
        # 4-8 plies random opening
        for _ in range(random.randint(4, 8)):
            if board.is_game_over():
                break
            board.push(random.choice(list(board.legal_moves)))

        while not board.is_game_over() and len(board.move_stack) < 100:
            info = engine.analyse(board, chess.engine.Limit(depth=15))
            score_cp = info["score"].white().score(mate_score=10000)
            best_move = info.get("pv", [None])[0]
            if score_cp is not None and best_move is not None:
                game_fens.append((board.fen(), score_cp, best_move.uci()))
                board.push(best_move)
            else:
                break

        res = board.result()
        result_val = 1 if res == "1-0" else (-1 if res == "0-1" else 0)
        for fen, score, move_uci in game_fens:
            f.write(f"{fen} | {score} | {result_val} | {move_uci}\n")
            total_positions += 1
        print(f"Game {game_idx + 1}/{NUM_GAMES} finished. Total positions: {total_positions}")

engine.quit()
print(f"\nGenerated {total_positions} positions saved to {POSITIONS_FILE}")

In [ ]:
# 3. Compile lightweight converter and build binpack
cpp_code = '''#include <iostream>
#include <fstream>
#include <string>
#include <sstream>
#include "data_loader/cpp/lib/chess.h"
#include "data_loader/cpp/lib/binpack.h"

using namespace chess;
using namespace binpack;

int main(int argc, char** argv) {
    if (argc < 3) return 1;
    std::ifstream in(argv[1]);
    if (!in.is_open()) return 1;
    CompressedTrainingDataEntryWriter writer(argv[2], std::ios_base::out | std::ios_base::trunc);
    std::string line;
    int count = 0;
    while (std::getline(in, line)) {
        if (line.empty()) continue;
        auto b1 = line.find('|');
        if (b1 == std::string::npos) continue;
        std::string fen = line.substr(0, b1);
        while (!fen.empty() && (fen.back() == ' ' || fen.back() == '\\t')) fen.pop_back();
        std::stringstream ss(line.substr(b1 + 1));
        int score = 0, result = 0;
        char sep;
        ss >> score;
        if (ss >> sep && sep == '|') ss >> result;
        TrainingDataEntry e;
        e.pos = Position::fromFen(fen);
        bool found = false;
        movegen::forEachLegalMove(e.pos, [&](Move m) {
            if (!found) { e.move = m; found = true; }
        });
        if (!found) continue;
        e.score = static_cast<std::int16_t>(score);
        e.ply = static_cast<std::uint16_t>(count % 100);
        e.result = static_cast<std::int16_t>(result);
        writer.addTrainingDataEntry(e);
        count++;
    }
    std::cout << "Packed " << count << " entries into " << argv[2] << std::endl;
    return 0;
}
'''
with open("fen2binpack.cpp", "w") as f:
    f.write(cpp_code)

!g++ -O3 -std=c++20 -I data_loader/cpp fen2binpack.cpp -o fen2binpack
!./fen2binpack generated_positions.txt custom_d15.binpack

In [ ]:
# 4. Train on the generated depth-15 dataset
!python train.py custom_d15.binpack \
    --features "Full_Threats+PP_3Wide+HalfKAv2_hm^+LatentThreats" \
    --batch-size 4096 \
    --epoch-size 10000 \
    --max-epochs 10 \
    --threads 4 \
    --default-root-dir ./logs/latent_threats_run